# Train YOLO Model with Kaggle Dataset on Google Colab

This notebook guides you step-by-step to configure the environment, download the soccer dataset from Kaggle, mount your Google Drive, and train the YOLO model.

### ⚠️ Important note before running:
Ensure you have enabled GPU in Colab to speed up training:
1. On the menu bar, select **Runtime** > **Change runtime type**.
2. Under **Hardware accelerator**, select **T4 GPU** (or any available GPU) and click **Save**.

## Step 1: Install required libraries
We need to install the `ultralytics` library to train YOLO and the `kaggle` library to download data via API.

In [ ]:
!pip install ultralytics kaggle

## Step 2: Configure Kaggle API to download the Dataset
To download data directly from Kaggle to Colab without downloading it to your local machine, you need the `kaggle.json` file (which contains your API Token).

**How to get the `kaggle.json` file:**
1. Log in to your [Kaggle](https://www.kaggle.com/) account.
2. Click on your profile picture in the top right corner > select **Settings**.
3. Scroll down to the **API** section and click **Create New Token**.
4. The `kaggle.json` file will automatically download to your computer.

Run the code cell below; an upload button will appear for you to select your downloaded `kaggle.json` file.

In [ ]:
from google.colab import files

# An upload dialog will appear, please select your kaggle.json file
files.upload()

# Set up the config directory and grant permissions for Kaggle API
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

Dataset URL: https://www.kaggle.com/datasets/enddl22/bounding-boxes-dflbundesliga-data-shootout
License(s): apache-2.0
  2% 261M/11.8G [00:16<12:13, 16.9MB/s]   
User cancelled operation


## Step 3: Download and extract the Dataset
Download the dataset from the link: `https://www.kaggle.com/datasets/enddl22/bounding-boxes-dflbundesliga-data-shootout` and extract it into the `datasets` folder in the Colab environment.

In [8]:
# Download the soccer dataset from Kaggle
!kaggle datasets download -d enddl22/bounding-boxes-dflbundesliga-data-shootout

# Extract data (-q to hide the file list for a cleaner output)
!unzip -q bounding-boxes-dflbundesliga-data-shootout.zip -d datasets/

# Check and print the exact location of the data.yaml config file
!find datasets/ -name "data.yaml"

Dataset URL: https://www.kaggle.com/datasets/enddl22/bounding-boxes-dflbundesliga-data-shootout
License(s): apache-2.0
bounding-boxes-dflbundesliga-data-shootout.zip: Skipping, found more recently modified local copy (use --force to force download)
replace datasets/yolo-soccer-pseudolabels/data.yaml? [y]es, [n]o, [A]ll, [N]one, [r]ename: datasets/yolo-soccer-pseudolabels/data.yaml


## Step 4: Mount Google Drive and Configure Model Training
The code cell below will connect Colab with your Google Drive. As the training progresses, all results, logs, and model weight files (`best.pt`, `last.pt`) will be **saved directly and continuously to your personal Drive**.

This ensures that even if Colab disconnects midway, your data remains completely safe and is not lost.

*Note: Change the `data="..."` path below if the `!find` command in Step 3 yields a different result.*

In [ ]:
from google.colab import drive
from ultralytics import YOLO
import os

# 1. Mount Google Drive
# An authentication prompt will appear, please click Agree / Allow
drive.mount('/content/drive')

# 2. Configure the storage directory on Google Drive
# Destination folder: My Drive > YOLO_Soccer_Train
save_dir = "/content/drive/MyDrive/YOLO_Soccer_Train"
os.makedirs(save_dir, exist_ok=True)

def train():
    # Initialize the model (Using the latest optimized yolo11n.pt, or change to yolov8n.pt if desired)
    model = YOLO("yolo11n.pt") 

    # Start training the model
    model.train(
        # Double-check if this path matches the result found in Step 3
        # data="/content/datasets/data.yaml",
        data="datasets/yolo-soccer-pseudolabels/data.yaml",  
        
        epochs=10,
        imgsz=960,           # Larger image size helps detect small/distant objects better (ball, players)
        batch=16,
        workers=8,
        device=0,            # Use the first GPU (T4 GPU)
        
        # CONFIGURATION TO SAVE TO GOOGLE DRIVE:
        project=save_dir,    # Save directly to the Google Drive folder created above
        name="soccer_run_1", # Name of the sub-directory to save results for this run
        save=True            # Enable automatic checkpoint saving
    )

if __name__ == "__main__":
    train()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ultralytics 8.4.61 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=datasets/yolo-soccer-pseudolabels/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.9

### 🏆 After completion
When the model finishes all epochs (or when you manually stop it), the best weights files will be located in your Google Drive at the following path:
`Your Drive` > `YOLO_Soccer_Train` > `soccer_run_1` > `weights` > `best.pt`.

Good luck with your model training!